# 🚀 BPE (Byte Pair Encoding) - The GPT Secret

## What is BPE?

**BPE is a subword tokenization algorithm that learns to split words into smaller, reusable pieces.**

### The Magic ✨

Instead of having separate tokens for every word:
```
❌ Word-level: "unhappiness" → one token (vocab = millions)
✅ BPE: "unhappiness" → ["un", "happiness"] (vocab = 50k)
```

### Why BPE Won 🏆

- **Small vocabulary**: 30k-50k tokens (vs millions for word-level)
- **No unknown words**: Can handle any word by breaking it down
- **Learns from data**: Automatically finds best subword units
- **Used by GPT**: GPT-2, GPT-3, GPT-4, RoBERTa all use BPE

## How BPE Works (The Algorithm)

### Step 1: Start with characters
```
Vocabulary: ['a', 'b', 'c', ..., 'z', ' ']
```

### Step 2: Find most frequent pair
```
Text: "low lower lowest"
Most frequent pair: 'l' + 'o' = 'lo'
```

### Step 3: Merge it
```
Add 'lo' to vocabulary
Text becomes: "lo w lo wer lo west"
```

### Step 4: Repeat
```
Next frequent: 'lo' + 'w' = 'low'
Text becomes: "low low er low est"
```

Keep going until you have desired vocabulary size!

## Let's Code It! 💻

## 1. BPE from Scratch (Understanding the Algorithm)

In [13]:
from collections import Counter, defaultdict
import re

def get_stats(vocab):
    """Count frequency of adjacent symbol pairs"""
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i + 1]] += freq
    return pairs

def merge_vocab(pair, vocab):
    """Merge most frequent pair in vocabulary"""
    vocab_out = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    
    for word in vocab:
        new_word = word.replace(bigram, replacement)
        vocab_out[new_word] = vocab[word]
    return vocab_out

# Sample corpus
corpus = [
    "low", "low", "low", "low",
    "lower", "lower", "lower",
    "newest", "newest",
    "widest"
]

# Step 1: Initialize vocabulary (split into characters)
vocab = {}
for word in corpus:
    word_with_spaces = ' '.join(list(word)) + ' </w>'  # </w> marks word end
    vocab[word_with_spaces] = vocab.get(word_with_spaces, 0) + 1

print("Initial vocabulary:")
for word, freq in sorted(vocab.items()):
    print(f"{word:20} : {freq}")

Initial vocabulary:
l o w </w>           : 4
l o w e r </w>       : 3
n e w e s t </w>     : 2
w i d e s t </w>     : 1


In [14]:
# Step 2: Learn BPE merges
num_merges = 10
merges = []

print("\nLearning BPE merges...\n")

for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        break
    
    # Find most frequent pair
    best_pair = max(pairs, key=pairs.get)
    merges.append(best_pair)
    
    print(f"Merge {i+1}: {best_pair} (frequency: {pairs[best_pair]})")
    
    # Merge the pair
    vocab = merge_vocab(best_pair, vocab)

print("\nFinal vocabulary after BPE:")
for word, freq in sorted(vocab.items()):
    print(f"{word:20} : {freq}")


Learning BPE merges...

Merge 1: ('l', 'o') (frequency: 7)
Merge 2: ('lo', 'w') (frequency: 7)
Merge 3: ('low', '</w>') (frequency: 4)
Merge 4: ('low', 'e') (frequency: 3)
Merge 5: ('lowe', 'r') (frequency: 3)
Merge 6: ('lower', '</w>') (frequency: 3)
Merge 7: ('e', 's') (frequency: 3)
Merge 8: ('es', 't') (frequency: 3)
Merge 9: ('est', '</w>') (frequency: 3)
Merge 10: ('n', 'e') (frequency: 2)

Final vocabulary after BPE:
low</w>              : 4
lower</w>            : 3
ne w est</w>         : 2
w i d est</w>        : 1


## 2. Using HuggingFace Tokenizers (Real-World BPE)

In [15]:
# Install if needed
# !pip install tokenizers

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Initialize a BPE tokenizer
tokenizer = Tokenizer(BPE(unk_token="<UNK>"))
tokenizer.pre_tokenizer = Whitespace()

# Create trainer
trainer = BpeTrainer(
    special_tokens=["<UNK>", "<PAD>", "<SOS>", "<EOS>"],
    vocab_size=1000,
    min_frequency=2
)

# Training corpus
training_corpus = [
    "The quick brown fox jumps over the lazy dog",
    "Natural language processing is amazing",
    "I love learning about tokenization",
    "Byte pair encoding is the foundation of GPT",
    "This is how modern AI models work",
    "Subword tokenization solves the vocabulary problem",
    "Machine learning and deep learning are transforming AI"
] * 10  # Repeat for better training

# Train the tokenizer
tokenizer.train_from_iterator(training_corpus, trainer=trainer)

print("✅ BPE Tokenizer trained!")
print(f"Vocabulary size: {tokenizer.get_vocab_size()}")




✅ BPE Tokenizer trained!
Vocabulary size: 175


In [16]:
# Test the tokenizer
test_sentences = [
    "The quick brown fox",
    "I love tokenization",
    "Supercalifragilisticexpialidocious",  # Unknown word!
    "preprocessing and postprocessing"
]

for sentence in test_sentences:
    output = tokenizer.encode(sentence)
    print(f"\nText: {sentence}")
    print(f"Tokens: {output.tokens}")
    print(f"IDs: {output.ids}")


Text: The quick brown fox
Tokens: ['The', 'quick', 'brown', 'fox']
IDs: [80, 161, 153, 135]

Text: I love tokenization
Tokens: ['I', 'love', 'tokenization']
IDs: [7, 106, 73]

Text: Supercalifragilisticexpialidocious
Tokens: ['Su', 'p', 'e', 'r', 'c', 'al', 'i', 'f', 'r', 'ag', 'i', 'l', 'is', 't', 'i', 'ce', 'x', 'p', 'i', 'al', 'i', 'do', 'c', 'io', 'u', 's']
IDs: [78, 28, 17, 30, 15, 83, 21, 18, 30, 81, 21, 24, 44, 32, 21, 88, 36, 28, 21, 83, 21, 93, 15, 50, 33, 31]

Text: preprocessing and postprocessing
Tokens: ['p', 'r', 'e', 'processing', 'and', 'p', 'o', 's', 't', 'processing']
IDs: [28, 30, 17, 171, 132, 28, 27, 31, 32, 171]


## 3. Using GPT-2 Tokenizer (Production-Ready)

In [17]:
# Install transformers
# !pip install transformers

from transformers import GPT2Tokenizer

# Load pre-trained GPT-2 tokenizer
tokenizer_gpt2 = GPT2Tokenizer.from_pretrained('gpt2')

print(f"GPT-2 Vocabulary size: {len(tokenizer_gpt2)}")
print(f"Special tokens: {tokenizer_gpt2.special_tokens_map}")

GPT-2 Vocabulary size: 50257
Special tokens: {'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}


In [18]:
# Test GPT-2 tokenizer
text = "I can't believe how amazing GPT's tokenization is! It handles unknown words like supercalifragilisticexpialidocious."

# Encode
tokens = tokenizer_gpt2.tokenize(text)
token_ids = tokenizer_gpt2.encode(text)

print(f"Text: {text}")
print(f"\nTokens ({len(tokens)}): {tokens}")
print(f"\nToken IDs: {token_ids}")

# Decode
decoded = tokenizer_gpt2.decode(token_ids)
print(f"\nDecoded: {decoded}")

# Notice:
# - "can't" becomes ["can", "'t"]
# - "supercalifragilisticexpialidocious" is broken into subwords
# - "GPT's" becomes ["G", "PT", "'s"]

Text: I can't believe how amazing GPT's tokenization is! It handles unknown words like supercalifragilisticexpialidocious.

Tokens (30): ['I', 'Ġcan', "'t", 'Ġbelieve', 'Ġhow', 'Ġamazing', 'ĠG', 'PT', "'s", 'Ġtoken', 'ization', 'Ġis', '!', 'ĠIt', 'Ġhandles', 'Ġunknown', 'Ġwords', 'Ġlike', 'Ġsuper', 'cal', 'if', 'rag', 'il', 'ist', 'ice', 'xp', 'ial', 'id', 'ocious', '.']

Token IDs: [40, 460, 470, 1975, 703, 4998, 402, 11571, 338, 11241, 1634, 318, 0, 632, 17105, 6439, 2456, 588, 2208, 9948, 361, 22562, 346, 396, 501, 42372, 498, 312, 32346, 13]

Decoded: I can't believe how amazing GPT's tokenization is! It handles unknown words like supercalifragilisticexpialidocious.


## 4. BPE vs Word Tokenization - Side by Side

In [19]:
# Compare approaches
test_words = [
    "unhappiness",
    "preprocessing",
    "supercalifragilisticexpialidocious",
    "COVID-19",
    "e-learning"
]

print("Word Tokenization vs BPE Comparison:\n")
print(f"{'Word':<45} {'Word-level':<20} {'BPE (GPT-2)'}")
print("="*90)

for word in test_words:
    # Word tokenization (just the word itself)
    word_tokens = [word]
    
    # BPE tokenization
    bpe_tokens = tokenizer_gpt2.tokenize(word)
    
    print(f"{word:<45} {str(word_tokens):<20} {str(bpe_tokens)}")

print("\n💡 Notice: BPE breaks unknown/complex words into known subwords!")

Word Tokenization vs BPE Comparison:

Word                                          Word-level           BPE (GPT-2)
unhappiness                                   ['unhappiness']      ['un', 'h', 'appiness']
preprocessing                                 ['preprocessing']    ['pre', 'processing']
supercalifragilisticexpialidocious            ['supercalifragilisticexpialidocious'] ['super', 'cal', 'if', 'rag', 'il', 'ist', 'ice', 'xp', 'ial', 'id', 'ocious']
COVID-19                                      ['COVID-19']         ['CO', 'VID', '-', '19']
e-learning                                    ['e-learning']       ['e', '-', 'learning']

💡 Notice: BPE breaks unknown/complex words into known subwords!


## 5. Vocabulary Comparison

In [20]:
# Analyze vocabulary efficiency
corpus_for_analysis = [
    "The runner is running in a running race",
    "I love lovely lovers who are loving",
    "Programming, programmer, and programs are related"
]

# Word-level vocabulary
word_vocab = set()
for sentence in corpus_for_analysis:
    word_vocab.update(sentence.lower().split())

# BPE vocabulary (approximate by unique tokens)
bpe_vocab = set()
for sentence in corpus_for_analysis:
    tokens = tokenizer_gpt2.tokenize(sentence.lower())
    bpe_vocab.update(tokens)

print("Vocabulary Size Comparison:")
print(f"Word-level unique tokens: {len(word_vocab)}")
print(f"BPE unique tokens: {len(bpe_vocab)}")

print(f"\nWord-level vocabulary: {sorted(word_vocab)}")
print(f"\nBPE vocabulary: {sorted(bpe_vocab)}")

print("\n🎯 BPE reuses subword pieces like 'ing', 'er', 'lov', reducing vocabulary!")

Vocabulary Size Comparison:
Word-level unique tokens: 19
BPE unique tokens: 21

Word-level vocabulary: ['a', 'and', 'are', 'i', 'in', 'is', 'love', 'lovely', 'lovers', 'loving', 'programmer,', 'programming,', 'programs', 'race', 'related', 'runner', 'running', 'the', 'who']

BPE vocabulary: [',', 'i', 'ming', 'program', 'the', 'Ġa', 'Ġand', 'Ġare', 'Ġin', 'Ġis', 'Ġlove', 'Ġlovely', 'Ġlovers', 'Ġloving', 'Ġprogrammer', 'Ġprograms', 'Ġrace', 'Ġrelated', 'Ġrunner', 'Ġrunning', 'Ġwho']

🎯 BPE reuses subword pieces like 'ing', 'er', 'lov', reducing vocabulary!


## 6. Practical: Tokenizing for RAG

In [21]:
# Real-world RAG example
documents = [
    "Retrieval-Augmented Generation (RAG) combines retrieval with generation.",
    "The retrieval component finds relevant documents from a knowledge base.",
    "The generation component uses an LLM to synthesize answers.",
    "RAG systems need efficient tokenization for both retrieval and generation."
]

print("Document Tokenization Analysis for RAG:\n")

for i, doc in enumerate(documents, 1):
    tokens = tokenizer_gpt2.encode(doc)
    print(f"Doc {i}: {doc[:60]}..." if len(doc) > 60 else f"Doc {i}: {doc}")
    print(f"  Token count: {len(tokens)}")
    print(f"  Tokens: {tokenizer_gpt2.tokenize(doc)[:10]}...\n")

# Calculate total tokens
total_tokens = sum(len(tokenizer_gpt2.encode(doc)) for doc in documents)
print(f"Total tokens in corpus: {total_tokens}")
print(f"Average tokens per document: {total_tokens/len(documents):.1f}")

Document Tokenization Analysis for RAG:

Doc 1: Retrieval-Augmented Generation (RAG) combines retrieval with...
  Token count: 15
  Tokens: ['Ret', 'ri', 'eval', '-', 'Aug', 'mented', 'ĠGeneration', 'Ġ(', 'RAG', ')']...

Doc 2: The retrieval component finds relevant documents from a know...
  Token count: 11
  Tokens: ['The', 'Ġretrieval', 'Ġcomponent', 'Ġfinds', 'Ġrelevant', 'Ġdocuments', 'Ġfrom', 'Ġa', 'Ġknowledge', 'Ġbase']...

Doc 3: The generation component uses an LLM to synthesize answers.
  Token count: 12
  Tokens: ['The', 'Ġgeneration', 'Ġcomponent', 'Ġuses', 'Ġan', 'ĠLL', 'M', 'Ġto', 'Ġsynthes', 'ize']...

Doc 4: RAG systems need efficient tokenization for both retrieval a...
  Token count: 12
  Tokens: ['RAG', 'Ġsystems', 'Ġneed', 'Ġefficient', 'Ġtoken', 'ization', 'Ġfor', 'Ġboth', 'Ġretrieval', 'Ġand']...

Total tokens in corpus: 50
Average tokens per document: 12.5


## 7. Token Limits and Chunking

In [22]:
# Understanding token limits for RAG
long_text = """Retrieval-Augmented Generation is a powerful technique that combines 
the strengths of retrieval systems and large language models. The process works by first 
retrieving relevant documents from a knowledge base, then using these documents as context 
for the language model to generate accurate and informed responses. This approach significantly 
improves the model's ability to provide factual information and reduces hallucinations."""

tokens = tokenizer_gpt2.encode(long_text)
print(f"Text length: {len(long_text)} characters")
print(f"Token count: {len(tokens)} tokens")
print(f"Ratio: {len(long_text)/len(tokens):.2f} characters per token")

# Chunking for different model limits
model_limits = {
    "BERT": 512,
    "GPT-3.5": 4096,
    "GPT-4": 8192,
    "Claude": 100000
}

print("\nChunking needs:")
for model, limit in model_limits.items():
    chunks_needed = (len(tokens) + limit - 1) // limit  # Ceiling division
    print(f"{model:10} (limit: {limit:6}): {chunks_needed} chunk(s) needed")

Text length: 436 characters
Token count: 79 tokens
Ratio: 5.52 characters per token

Chunking needs:
BERT       (limit:    512): 1 chunk(s) needed
GPT-3.5    (limit:   4096): 1 chunk(s) needed
GPT-4      (limit:   8192): 1 chunk(s) needed
Claude     (limit: 100000): 1 chunk(s) needed


## 8. Advanced: Byte-Level BPE (GPT-2 style)

In [23]:
# GPT-2 uses byte-level BPE for universal coverage
multilingual_text = [
    "Hello, world!",           # English
    "¡Hola, mundo!",           # Spanish
    "Bonjour, le monde!",      # French
    "こんにちは世界",             # Japanese
    "🚀 Emoji test 🎉",        # Emojis
]

print("Byte-Level BPE handles ANY text:\n")

for text in multilingual_text:
    tokens = tokenizer_gpt2.tokenize(text)
    token_count = len(tokens)
    print(f"{text:30} → {token_count:2} tokens: {tokens}")

print("\n✨ Byte-level BPE can tokenize ANY Unicode text!")

Byte-Level BPE handles ANY text:

Hello, world!                  →  4 tokens: ['Hello', ',', 'Ġworld', '!']
¡Hola, mundo!                  →  8 tokens: ['Â', '¡', 'H', 'ola', ',', 'Ġmund', 'o', '!']
Bonjour, le monde!             →  8 tokens: ['Bon', 'j', 'our', ',', 'Ġle', 'Ġm', 'onde', '!']
こんにちは世界                        → 10 tokens: ['ãģĵ', 'ãĤĵ', 'ãģ«', 'ãģ', '¡', 'ãģ¯', 'ä¸', 'ĸ', 'çķ', 'Į']
🚀 Emoji test 🎉                 →  9 tokens: ['ðŁ', 'ļ', 'Ģ', 'ĠEm', 'oji', 'Ġtest', 'ĠðŁ', 'İ', 'ī']

✨ Byte-level BPE can tokenize ANY Unicode text!


## 9. Building Custom BPE for Your Domain

In [24]:
# Train domain-specific BPE tokenizer
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

# Medical domain example
medical_corpus = [
    "The patient presents with hypertension and hyperlipidemia.",
    "Prescribed antihypertensive medication for blood pressure control.",
    "Cardiovascular examination reveals normal heart sounds.",
    "Electrocardiogram shows sinus rhythm without abnormalities.",
    "Diagnosis: Essential hypertension, hypercholesterolemia."
] * 20  # Repeat for better learning

# Create custom BPE tokenizer
custom_tokenizer = Tokenizer(models.BPE())
custom_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

trainer = trainers.BpeTrainer(
    vocab_size=500,
    special_tokens=["<UNK>", "<PAD>", "<SOS>", "<EOS>"]
)

custom_tokenizer.train_from_iterator(medical_corpus, trainer=trainer)

# Test on medical text
test_medical = "Patient with hypertensive cardiovascular disease and hyperlipidemia."
output = custom_tokenizer.encode(test_medical)

print("Custom Medical BPE Tokenizer:")
print(f"Text: {test_medical}")
print(f"Tokens: {output.tokens}")
print("\n💡 Notice: Medical terms are tokenized efficiently!")




Custom Medical BPE Tokenizer:
Text: Patient with hypertensive cardiovascular disease and hyperlipidemia.
Tokens: ['P', 'ati', 'ent', 'with', 'hyperten', 'sive', 'cardio', 'vas', 'cular', 'd', 'i', 's', 'e', 'as', 'e', 'and', 'hyperlipidemia', '.']

💡 Notice: Medical terms are tokenized efficiently!


## Key Takeaways 🎯

### ✅ Why BPE is Revolutionary:

1. **Efficient Vocabulary**: 30k-50k tokens vs millions for word-level
2. **No Unknown Words**: Can handle any word by breaking it down
3. **Data-Driven**: Learns optimal subwords from your corpus
4. **Universal**: Byte-level BPE handles any language/emoji

### 🔑 Key Concepts:

- **Merge Rules**: BPE learns to merge frequent character pairs
- **Subword Units**: Balance between characters and words
- **Compression**: Reuses subword pieces across vocabulary
- **Flexibility**: Adapts to different domains and languages

### 🚀 For RAG Systems:

**Use BPE when:**
- Using GPT models (GPT-2, GPT-3, GPT-4)
- Need multilingual support
- Working with technical/domain-specific text
- Want to minimize vocabulary size

**Considerations:**
- Match tokenizer to your embedding model
- Consider token limits when chunking
- Train custom tokenizer for specialized domains

### 📊 BPE in Production:

```python
# For RAG with GPT models:
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Tokenize documents
tokens = tokenizer.encode(document)

# Check token count for chunking
if len(tokens) > 4096:
    # Chunk the document
    pass
```

## Next Steps 🔜

Move to `03_WordPiece_Tokenization.ipynb` to learn BERT's approach!

That's the tokenizer used by the world's most popular embedding models! 🎓